# VisDrone Detection — Antlings Internship Assessment

End-to-end pipeline: Dataset → Training → Detection → Evaluation

In [ ]:
# Step 1: Install
!pip install ultralytics lap -q
import ultralytics
ultralytics.checks()

In [ ]:
# Step 2: Clone repo
!git clone https://github.com/MahfuzAhemedShimul/visdrone-detection.git
%cd visdrone-detection

In [ ]:
# Step 3: Verify dataset path
import os
BASE = '/kaggle/input/datasets/banuprasadb/visdrone-dataset/VisDrone_Dataset'
print('Train images:', len(os.listdir(f'{BASE}/VisDrone2019-DET-train/images')))
print('Val images  :', len(os.listdir(f'{BASE}/VisDrone2019-DET-val/images')))

In [ ]:
# Step 4: Task 1 — Dataset visualization
import sys
sys.path.insert(0, '.')
from prepare_dataset import visualize_samples
from pathlib import Path

visualize_samples(
    img_dir   = f'{BASE}/VisDrone2019-DET-train/images',
    lbl_dir   = f'{BASE}/VisDrone2019-DET-train/labels',
    n         = 6,
    save_path = 'outputs/sample_visualizations.png'
)

In [ ]:
# Step 5: Write data.yaml
import yaml
from pathlib import Path

OUTPUT = Path('/kaggle/working/visdrone_yolo')
OUTPUT.mkdir(exist_ok=True)

yaml_content = {
    'path' : BASE,
    'train': 'VisDrone2019-DET-train/images',
    'val'  : 'VisDrone2019-DET-val/images',
    'nc'   : 10,
    'names': ['pedestrian','people','bicycle','car','van',
              'truck','tricycle','awning-tricycle','bus','motor']
}
with open(OUTPUT / 'data.yaml', 'w') as f:
    yaml.dump(yaml_content, f)

print('✅ data.yaml ready!')

In [ ]:
# Step 6: Task 2 — Train YOLOv8m (50 epochs ~1.5 hrs)
from ultralytics import YOLO

model = YOLO('yolov8m.pt')
model.train(
    data     = '/kaggle/working/visdrone_yolo/data.yaml',
    epochs   = 50,
    imgsz    = 640,
    batch    = 16,
    mosaic   = 1.0,
    patience = 15,
    project  = '/kaggle/working/runs/visdrone',
    name     = 'yolov8m_exp',
    device   = 0,
    workers  = 2,
)
print('✅ Training complete!')

In [ ]:
# Step 7: Show training curves
from IPython.display import Image
Image('/kaggle/working/runs/visdrone/yolov8m_exp/results.png')

In [ ]:
# Step 8: Task 3 — Detection + Human Counting
from ultralytics import YOLO
from pathlib import Path
import cv2, matplotlib.pyplot as plt

WEIGHTS = '/kaggle/working/runs/visdrone/yolov8m_exp/weights/best.pt'
model   = YOLO(WEIGHTS)

PERSON_CLASSES = {0, 1}   # pedestrian, people
CAR_CLASSES    = {3, 4}   # car, van

val_imgs = list(Path(f'{BASE}/VisDrone2019-DET-val/images').glob('*.jpg'))[:6]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Human & Car Detection with Counting', fontsize=14)

for ax, img_path in zip(axes.flatten(), val_imgs):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    results = model(img, conf=0.25, verbose=False)[0]

    persons = cars = 0
    for box in results.boxes:
        cls = int(box.cls[0])
        x1,y1,x2,y2 = map(int, box.xyxy[0])
        if cls in PERSON_CLASSES:
            cv2.rectangle(img,(x1,y1),(x2,y2),(57,255,20),2)
            persons += 1
        elif cls in CAR_CLASSES:
            cv2.rectangle(img,(x1,y1),(x2,y2),(255,144,30),2)
            cars += 1

    cv2.putText(img,f'Humans: {persons}',(10,35),
                cv2.FONT_HERSHEY_SIMPLEX,1.0,(57,255,20),2)
    cv2.putText(img,f'Cars: {cars}',(10,70),
                cv2.FONT_HERSHEY_SIMPLEX,1.0,(255,144,30),2)

    ax.imshow(img)
    ax.set_title(f'Humans:{persons}  Cars:{cars}', fontsize=9)
    ax.axis('off')

plt.tight_layout()
Path('outputs').mkdir(exist_ok=True)
plt.savefig('outputs/detection_results.png', dpi=150)
plt.show()
print('✅ Detection done!')

In [ ]:
# Step 9: Task 5 — Evaluation
metrics = model.val(
    data  = '/kaggle/working/visdrone_yolo/data.yaml',
    split = 'val',
)
print(f'\nmAP@0.5      : {metrics.box.map50:.4f}')
print(f'mAP@0.5:0.95 : {metrics.box.map:.4f}')
print(f'Precision    : {metrics.box.mp:.4f}')
print(f'Recall       : {metrics.box.mr:.4f}')

In [ ]:
# Step 10: Copy outputs and zip for download
import shutil
from pathlib import Path

out = Path('outputs')
out.mkdir(exist_ok=True)

shutil.copy('/kaggle/working/runs/visdrone/yolov8m_exp/results.png', out)
shutil.copy('/kaggle/working/runs/visdrone/yolov8m_exp/val_batch0_pred.jpg', out)

shutil.make_archive('/kaggle/working/final_outputs', 'zip', out)
print('✅ Download final_outputs.zip from the Output panel →')